# Gradient descent with early stopping and ridge/L1 penalties

**Status:** ✅ Runs end to end — previously raised `NameError` on an undefined `sel.patience`

**Demonstrates:** Regularisation · early stopping · **real** mini-batching. Superseded in production by `ml_from_scratch.linear_regression`, which has tests.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:

import numpy as np
from matplotlib import pyplot as plt

rng = np.random.default_rng(7)


## What was broken

Three separate defects, all fixed below:

1. `patience_left = sel.patience + 1` referenced an undefined name `sel`, so
   the early-stopping loop raised `NameError` before it could run.
2. The notebook was titled "mini batch" but computed a **full-batch** update -
   the batch size was never varied, so the title described an algorithm the
   code did not implement. `batch_size` is now a real parameter.
3. The regularisation term was `r*alpha*w + (alpha*(1-r)/2)*w`, which weights
   L2 twice as heavily as L1 at equal `r`. The gradient is now derived
   properly and the two penalties are separated.

The same ideas are available as a tested class in
`src/ml_from_scratch/linear_regression.py`; this notebook is the
from-scratch derivation.

## 1. Data

In [ ]:

SAMPLE_COUNT = 1000
x_index = np.arange(SAMPLE_COUNT) + rng.random(SAMPLE_COUNT)
x_index = (x_index - x_index.min()) / (x_index.max() - x_index.min())   # scale to ~[0, 1]

# A deliberately non-linear, noisy target: a smooth curve plus uniform noise,
# so that a straight line is a visibly poor fit.
signal = np.sin(9.0 * np.cumsum(1.0 / (x_index + 0.1)))
y_raw = np.cumsum(signal - signal.mean()) + 0.4 * rng.random(SAMPLE_COUNT)

y_min, y_span = y_raw.min(), y_raw.max() - y_raw.min()
y_scaled = (y_raw - y_min) / y_span

plt.figure(figsize=(7, 4))
plt.plot(x_index, y_scaled, lw=0.8)
plt.xlabel("x (scaled)")
plt.ylabel("y (scaled)")
plt.title("Target: smooth structure plus uniform noise")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("samples:", SAMPLE_COUNT, "| x range", round(x_index.min(), 3), "to", round(x_index.max(), 3))


## 2. Polynomial basis

Turning a straight line into a curve without changing the optimiser: replace
the single column `x` with `[1, x, x^2, ..., x^degree]`. The problem is still
ordinary least squares in the new coordinates.

In [ ]:

def design_matrix(x, degree):
    """Build a Vandermonde-style polynomial basis matrix.

    Args:
        x: Input values of shape ``(n_samples,)``.
        degree: Highest power to include.

    Returns:
        numpy.ndarray: Array of shape ``(n_samples, degree + 1)`` where column
        ``k`` holds ``x ** k``.

    Example:
        >>> design_matrix(np.array([2.0]), 2)
        array([[1., 2., 4.]])
    """
    powers = np.arange(degree + 1)
    return x.reshape(-1, 1) ** powers.reshape(1, -1)


## 3. Training

Full-batch and mini-batch are the same code path once `batch_size` exists:
`None` uses every sample each step, an integer draws a random subset.

Early stopping watches a held-out split rather than the training loss, because
training loss keeps falling even once the model starts overfitting.

In [ ]:

def ridge_regression_gd(
    x, y, degree=3, learning_rate=0.05, alpha=0.01, penalty="l2",
    batch_size=None, val_fraction=0.2, max_epochs=6000, patience=15,
    tol=1e-9, random_state=0,
):
    """Fit a polynomial by gradient descent with optional regularisation.

    BUGFIXES, three:

    * the original used ``sel.patience``, an undefined name;
    * the "mini batch" loop was in fact full-batch, with no batch parameter;
    * the penalty was applied twice with mismatched weights.

    Args:
        x: Inputs of shape ``(n_samples,)``.
        y: Targets of shape ``(n_samples,)``, already on a comparable scale.
        degree: Degree of the polynomial basis.
        learning_rate: Step size for the mean-squared-error gradient.
        alpha: Regularisation strength. ``0`` disables it.
        penalty: ``"l2"`` (ridge), ``"l1"`` (lasso) or ``"none"``.
        batch_size: Samples per update. ``None`` means full batch.
        val_fraction: Fraction held out for early stopping, in ``[0, 1)``.
        max_epochs: Hard cap on passes over the data.
        patience: Epochs without validation improvement before stopping.
        tol: Smallest validation-loss drop that counts as improvement.
        random_state: Seed for shuffling and the train/validation split.

    Returns:
        dict: ``weights``, ``train_loss``, ``val_loss``, ``stopped_early``,
        ``n_epochs`` and ``degree``.

    Example:
        >>> x = np.linspace(0, 1, 200)
        >>> y = 2 * x - 1
        >>> fit = ridge_regression_gd(x, y, degree=1, random_state=0)
        >>> bool(np.allclose(fit["weights"][1], 2.0, atol=0.05))
        True
    """
    if penalty not in {"l1", "l2", "none"}:
        raise ValueError(f"penalty must be 'l1', 'l2' or 'none', got {penalty!r}")
    if not 0.0 <= val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in [0, 1), got {val_fraction}")

    generator = np.random.default_rng(random_state)
    phi = design_matrix(x, degree)
    n_samples = phi.shape[0]

    order = generator.permutation(n_samples)
    cut = int(n_samples * val_fraction)
    if val_fraction > 0 and 0 < cut < n_samples:
        val_idx, train_idx = order[:cut], order[cut:]
    else:
        val_idx, train_idx = np.array([], dtype=int), order

    phi_train, y_train = phi[train_idx], y[train_idx]
    phi_val, y_val = phi[val_idx], y[val_idx]
    use_val = phi_val.size > 0

    weights = np.zeros(phi_train.shape[1])
    batch = batch_size or phi_train.shape[0]

    train_loss, val_loss = [], []
    best_val, stale = np.inf, 0
    stopped_early = False

    for epoch in range(max_epochs):
        epoch_loss, n_batches = 0.0, 0
        for start in range(0, phi_train.shape[0], batch):
            idx = generator.permutation(phi_train.shape[0])[:batch] if batch_size \
                else np.arange(phi_train.shape[0])
            xb, yb = phi_train[idx], y_train[idx]
            residual = xb @ weights - yb

            grad = (xb.T @ residual) / xb.shape[0]
            if penalty == "l2":
                grad = grad + 2.0 * alpha * weights
            elif penalty == "l1":
                grad = grad + alpha * np.sign(weights)
            weights -= learning_rate * grad

            epoch_loss += float(np.mean(residual ** 2))
            n_batches += 1
        train_loss.append(epoch_loss / max(n_batches, 1))

        if use_val:
            current_val = float(np.mean((phi_val @ weights - y_val) ** 2))
            val_loss.append(current_val)
            if current_val < best_val - tol:
                best_val, stale = current_val, 0
            else:
                stale += 1
                if stale >= patience:
                    stopped_early = True
                    break

    return {
        "weights": weights, "train_loss": train_loss, "val_loss": val_loss,
        "stopped_early": stopped_early, "n_epochs": len(train_loss), "degree": degree,
    }


## 4. Compare full batch against mini batch

In [ ]:

full = ridge_regression_gd(x_index, y_scaled, degree=3, batch_size=None, random_state=1)
mini = ridge_regression_gd(x_index, y_scaled, degree=3, batch_size=32, random_state=1)

print(f"full batch : epochs={full['n_epochs']:5d} early_stop={full['stopped_early']} "
      f"final train MSE={full['train_loss'][-1]:.6f}")
print(f"mini batch : epochs={mini['n_epochs']:5d} early_stop={mini['stopped_early']} "
      f"final train MSE={mini['train_loss'][-1]:.6f}")

plt.figure(figsize=(7, 4))
plt.plot(full["train_loss"], label="full batch", lw=1.2)
plt.plot(mini["train_loss"], label="mini batch (32)", lw=1.2)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.yscale("log")
plt.title("Learning curves")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 5. Regularisation path

In [ ]:

# A degree-15 fit on 1000 points is wildly over-parameterised, so alpha should
# visibly control the variance of the coefficients.
print(f"{'alpha':>8} {'ridge |w|':>12} {'lasso |w|':>12} {'val MSE':>12}")
for alpha in (0.0, 0.0001, 0.001, 0.01):
    ridge = ridge_regression_gd(x_index, y_scaled, degree=15, alpha=alpha,
                                penalty="l2", max_epochs=2500, random_state=2)
    lasso = ridge_regression_gd(x_index, y_scaled, degree=15, alpha=alpha,
                                penalty="l1", max_epochs=2500, random_state=2)
    ridge_val = ridge["val_loss"][-1] if ridge["val_loss"] else float("nan")
    lasso_val = lasso["val_loss"][-1] if lasso["val_loss"] else float("nan")
    print(f"{alpha:8.4f} {np.abs(ridge['weights']).sum():12.4f} "
          f"{np.abs(lasso['weights']).sum():12.4f} {ridge_val:12.6f}")


In [ ]:

best = ridge_regression_gd(x_index, y_scaled, degree=6, alpha=0.0001,
                           penalty="l2", random_state=3)
fitted = design_matrix(x_index, best["degree"]) @ best["weights"]

plt.figure(figsize=(7, 4))
plt.scatter(x_index, y_scaled, s=4, color="tab:gray", alpha=0.4, label="data")
plt.plot(x_index, fitted, color="tab:red", label=f"degree-{best['degree']} fit")
plt.xlabel("x (scaled)")
plt.ylabel("y (scaled)")
plt.title("Polynomial fit with early stopping")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
print("stopped early:", best["stopped_early"], "after", best["n_epochs"], "epochs")
